In [4]:
import torch
import torch.nn as nn
from torchvision import transforms
import timm
from PIL import Image
import os
from huggingface_hub import hf_hub_download

# Download the model file from Hugging Face
print("Downloading model...")
model_path = hf_hub_download(repo_id="VisionaryQuant/5_Crop_Disease_Detection", filename="best_crop_disease_model.pt")

# Load the model architecture - EfficientNet-B3 matches the dataset of 17 classes
model = timm.create_model('efficientnet_b3', pretrained=False)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier.in_features, 17)
)

# Load the state dict
print("Loading model weights...")
state_dict = torch.load(model_path, map_location=torch.device('cpu'))
model.load_state_dict(state_dict)
model.eval()

# Preprocess input image
image_path = "images.jpeg"
if not os.path.exists(image_path):
    print(f"Note: Please provide an image named '{image_path}' to test the inference!")
else:
    image = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((300, 300)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    input_tensor = transform(image).unsqueeze(0)

    # Run inference
    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.nn.functional.softmax(logits, dim=1)
        predicted_idx = torch.argmax(probs, dim=1).item()

    # The repo model is trained on 17 classes, not 33
    CLASS_NAMES = [
        "Corn___Common_Rust", "Corn___Gray_Leaf_Spot", "Corn___Healthy", "Corn___Northern_Leaf_Blight",
        "Potato___Early_Blight", "Potato___Healthy", "Potato___Late_Blight",
        "Rice___Brown_Spot", "Rice___Healthy", "Rice___Leaf_Blast", "Rice___Neck_Blast",
        "Sugarcane___Bacterial_Blight", "Sugarcane___Healthy", "Sugarcane___Red_Rot",
        "Wheat___Brown_Rust", "Wheat___Healthy", "Wheat___Yellow_Rust"
    ]
    
    if predicted_idx < len(CLASS_NAMES):
        predicted_label = CLASS_NAMES[predicted_idx]
    else:
        predicted_label = f"Unknown class index: {predicted_idx}"
        
    print(f"Predicted class index: {predicted_idx}")
    print(f"Predicted label: {predicted_label}")


Loading model weights...
Predicted class index: 13
Predicted label: Sugarcane___Red_Rot
